![image.png](https://i.imgur.com/a3uAqnb.png)

# Semantic Search in a Refinery Manual

- **Dataset**: a 50-page refinery operations guide (one row per page: page number, section, text) and 10 technician questions with the pages an engineer would want, in English and Arabic
- **Task**: retrieve the right pages for a question, and measure how many correct pages each method finds in its top 5
- **Model**: averaged GloVe word vectors, a TF-IDF baseline, and two pretrained sentence-embedding models (English-only and multilingual), all used as they are
- **Runtime**: CPU (the default Colab runtime), about 5 minutes including downloads

**Scenario:** a technician types a question like *"pump vibration too high"*. Which pages of the guide should come back?

In this lab, we will:

- turn each page and each question into a vector, and rank pages by cosine similarity
- compare raw and preprocessed text with averaged GloVe vectors
- add a TF-IDF keyword baseline and sentence embeddings, and score all methods on the same ground truth
- ask the same questions in Arabic and see which embedding model can still find the English pages

This notebook is adapted from *Semantic Search in a Refinery Manual* by Yazan alshoibi; its tokenizer tasks are now guided steps, and Steps 8 to 10 are new.

## Setup

The default Colab runtime (CPU) is enough; run the cells from top to bottom. `gensim` downloads the GloVe vectors; everything else ships with Colab.

In [1]:
%pip install -q gensim

In [2]:
import gensim.downloader as api
import nltk
import numpy as np
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

SEED = 42
TOP_K = 5                                   # pages returned for each question
GUIDE_URL = ("https://raw.githubusercontent.com/KAUST-Academy/2026_Tahakom_Advanced_Natural_Language_Processing/"
             "main/Labs/Day_4/refinery_operations_guide.xlsx")
ENGLISH_EMBEDDER = "sentence-transformers/all-MiniLM-L6-v2"
MULTILINGUAL_EMBEDDER = "intfloat/multilingual-e5-small"
np.random.seed(SEED)

Load the GloVe word vectors: 400,000 words, 100 numbers each, trained on Wikipedia and newswire text. The download is about 130 MB.

In [3]:
model = api.load("glove-wiki-gigaword-100")
print(f"{len(model):,} words, {model.vector_size} dimensions")

[==================================================] 100.0% 128.1/128.1MB downloaded
400,000 words, 100 dimensions


## Step 1: Load the Guide

The guide is an Excel sheet with one row per page. `pandas` reads it straight from its web address.

In [4]:
guide = pd.read_excel(GUIDE_URL)
print(f"Loaded {len(guide)} pages\n")
guide.head()

Loaded 50 pages



,page_number,section,content
0,1,Equipment Overview,This chapter provides an overview of the major...
1,2,Centrifugal Pumps,Centrifugal pumps are the most common type of ...
2,3,Pump Maintenance,Routine pump maintenance includes vibration mo...
3,4,Pump Troubleshooting,"Common pump problems include cavitation, seal ..."
4,5,Compressors,Reciprocating and centrifugal compressors are ...


In [5]:
# See all sections, then read one page
for _, row in guide.iterrows():
    print(f"  Page {row['page_number']:>2}: {row['section']}")
print()
print(guide.loc[3, "content"])

  Page  1: Equipment Overview
  Page  2: Centrifugal Pumps
  Page  3: Pump Maintenance
  Page  4: Pump Troubleshooting
  Page  5: Compressors
  Page  6: Compressor Lubrication
  Page  7: Heat Exchangers
  Page  8: Heat Exchanger Cleaning
  Page  9: Control Valves
  Page 10: Electric Motors
  Page 11: Crude Distillation Unit
  Page 12: Vacuum Distillation Unit
  Page 13: Hydrocracker Unit
  Page 14: Diesel Hydrotreater
  Page 15: Naphtha Hydrotreater
  Page 16: Sulfur Recovery Unit
  Page 17: Hydrogen Production Unit
  Page 18: Amine Treating Unit
  Page 19: Sour Water Stripper
  Page 20: Tank Farm Operations
  Page 21: Pressure Transmitters
  Page 22: Level Transmitters
  Page 23: Flow Transmitters
  Page 24: Temperature Transmitters
  Page 25: DCS and Control Systems
  Page 26: Instrument Calibration
  Page 27: Analyzer Systems
  Page 28: Emergency Shutdown System
  Page 29: Fire and Gas Detection
  Page 30: H2S Safety
  Page 31: Relief Valves
  Page 32: Flare System
  Page 33: Person

## Step 2: Tokenize the Text

In [6]:
def simple_tokenize(text):
    return text.lower().split()

## Step 3: Turn a Page into One Vector

A page vector is the average of the GloVe vectors of its words. Words that GloVe does not know are skipped.

In [7]:
def page_vector(text, embeddings, tokenize_fn):
    # Split text into words
    words = tokenize_fn(text)

    # Look up each word in GloVe (skip unknown words)
    vectors = []
    for word in words:
        if word in embeddings:
            vectors.append(embeddings[word])

    # Average all vectors into one
    if len(vectors) == 0:
        return np.zeros(100)
    return np.mean(vectors, axis=0)

## Step 4: Build Vectors for All 50 Pages

In [8]:
raw_vectors = []

for text in guide["content"]:
    vec = page_vector(text, model, simple_tokenize)
    raw_vectors.append(vec)

raw_vectors = np.array(raw_vectors)

print(f"Shape: {raw_vectors.shape}")
print("(50 pages x 100 dimensions: each page is now one row of numbers)")

Shape: (50, 100)
(50 pages x 100 dimensions: each page is now one row of numbers)


## Step 5: Search
Given a question, we:
1. Turn the question into a vector (same way as pages)
2. Compare it to every page vector using **cosine similarity**
3. Return the top 5 most similar pages

Cosine similarity: **1.0** = same direction, **0.0** = unrelated.

In [9]:
def search(query, page_vectors, tokenize_fn, top_n=TOP_K):
    # Turn query into a vector
    query_vec = page_vector(query, model, tokenize_fn)
    query_vec = query_vec.reshape(1, -1)   # from (100,) to (1, 100)

    # Compare with every page
    scores = cosine_similarity(query_vec, page_vectors)[0]   # shape (50,)

    # Sort by highest score
    top_indices = scores.argsort()[::-1][:top_n]   # argsort sorts ascending; [::-1] reverses it

    # Print results
    print(f'Query: "{query}"\n')
    for rank, idx in enumerate(top_indices, 1):
        page = guide.loc[idx, "page_number"]
        section = guide.loc[idx, "section"]
        score = scores[idx]
        print(f"   {rank}. Page {page:>2}: {section:<35s}  (score: {score:.3f})")

    return top_indices

In [10]:
for query in ["pump vibration is too high", "H2S gas detected in the area", "how to calibrate a pressure transmitter"]:
    search(query, raw_vectors, simple_tokenize)
    print()

Query: "pump vibration is too high"

   1. Page 30: H2S Safety                           (score: 0.895)
   2. Page 47: Steam System                         (score: 0.888)
   3. Page  3: Pump Maintenance                     (score: 0.880)
   4. Page  4: Pump Troubleshooting                 (score: 0.878)
   5. Page 17: Hydrogen Production Unit             (score: 0.876)

Query: "H2S gas detected in the area"

   1. Page 29: Fire and Gas Detection               (score: 0.912)
   2. Page 19: Sour Water Stripper                  (score: 0.895)
   3. Page 18: Amine Treating Unit                  (score: 0.891)
   4. Page 16: Sulfur Recovery Unit                 (score: 0.891)
   5. Page 32: Flare System                         (score: 0.889)

Query: "how to calibrate a pressure transmitter"

   1. Page 21: Pressure Transmitters                (score: 0.919)
   2. Page 28: Emergency Shutdown System            (score: 0.903)
   3. Page 44: Abnormal Situation Management        (score: 0.902)
 

## Step 6: Why Preprocessing Might Help

Look at the query *"H2S gas detected in the area what to do"*:

```
["h2s", "gas", "detected", "in", "the", "area", "what", "to", "do"]
```

The words **"in", "the", "what", "to", "do"** appear on every page. They add noise and push the vector away from the right answer.

**Solution:** remove stop words and lemmatize before building vectors.

```
Before: ["h2s", "gas", "detected", "in", "the", "area", "what", "to", "do"]
After:  ["h2s", "gas", "detected", "area"]
```

In [11]:
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

### A clean tokenizer

The clean tokenizer keeps only alphabetic words, drops stop words, and lemmatizes what is left (``pumps`` becomes ``pump``).

In [12]:
def clean_tokenize(text):
    words = text.lower().split()
    cleaned = []
    for word in words:
        if not word.isalpha():                 # skip numbers and punctuation
            continue
        if word in stop_words:                 # skip stop words
            continue
        word = lemmatizer.lemmatize(word)      # lemmatize the word
        cleaned.append(word)
    return cleaned

sample = "The pumps were leaking and operators isolated the equipment"
print("Raw:    ", simple_tokenize(sample))
print("Cleaned:", clean_tokenize(sample))

Raw:     ['the', 'pumps', 'were', 'leaking', 'and', 'operators', 'isolated', 'the', 'equipment']
Cleaned: ['pump', 'leaking', 'operator', 'isolated', 'equipment']


### Clean page vectors

The same averaging as before, with the clean tokenizer.

In [13]:
clean_vectors = np.array([page_vector(text, model, clean_tokenize) for text in guide["content"]])
print(f"Shape: {clean_vectors.shape}")

Shape: (50, 100)


### Search and compare

The same two queries, without and with preprocessing.

In [14]:
for query in ["pump vibration is too high", "H2S gas detected in the area what to do"]:
    print("== WITHOUT preprocessing ==")
    search(query, raw_vectors, simple_tokenize)
    print("\n== WITH preprocessing ==")
    search(query, clean_vectors, clean_tokenize)
    print("\n" + "-" * 70 + "\n")

== WITHOUT preprocessing ==
Query: "pump vibration is too high"

   1. Page 30: H2S Safety                           (score: 0.895)
   2. Page 47: Steam System                         (score: 0.888)
   3. Page  3: Pump Maintenance                     (score: 0.880)
   4. Page  4: Pump Troubleshooting                 (score: 0.878)
   5. Page 17: Hydrogen Production Unit             (score: 0.876)

== WITH preprocessing ==
Query: "pump vibration is too high"

   1. Page  5: Compressors                          (score: 0.814)
   2. Page 47: Steam System                         (score: 0.813)
   3. Page  3: Pump Maintenance                     (score: 0.806)
   4. Page 13: Hydrocracker Unit                    (score: 0.799)
   5. Page  4: Pump Troubleshooting                 (score: 0.797)

----------------------------------------------------------------------

== WITHOUT preprocessing ==
Query: "H2S gas detected in the area what to do"

   1. Page 29: Fire and Gas Detection              

## Step 7: Measure with Ground Truth

So far we have been eyeballing results. Now we measure.

We have **10 queries with correct answers**: the pages a refinery engineer would want. For every method we count **how many correct pages it finds in its top 5**. Each query also has an Arabic version, used in Step 10.

In [15]:
queries_with_answers = [
    {"query": "pump seal is leaking oil", "query_ar": "مانع التسرب في المضخة يسرب الزيت",
     "golden_pages": [2, 3, 4], "why": "Centrifugal Pumps, Pump Maintenance, Pump Troubleshooting"},
    {"query": "H2S gas detected in the area what to do", "query_ar": "تم رصد غاز كبريتيد الهيدروجين في المنطقة، ماذا أفعل؟",
     "golden_pages": [30, 29, 33], "why": "H2S Safety, Fire and Gas Detection, PPE"},
    {"query": "how to calibrate a pressure transmitter", "query_ar": "كيف أعاير جهاز إرسال الضغط؟",
     "golden_pages": [21, 26], "why": "Pressure Transmitters, Instrument Calibration"},
    {"query": "relief valve opened and gas is going to flare", "query_ar": "انفتح صمام التنفيس والغاز يذهب إلى الشعلة",
     "golden_pages": [31, 32], "why": "Relief Valves, Flare System"},
    {"query": "electric motor keeps tripping the breaker", "query_ar": "المحرك الكهربائي يفصل القاطع باستمرار",
     "golden_pages": [10, 35], "why": "Electric Motors, LOTO Procedures"},
    {"query": "heat exchanger pressure drop is increasing", "query_ar": "انخفاض الضغط في المبادل الحراري يزداد",
     "golden_pages": [7, 8], "why": "Heat Exchangers, Heat Exchanger Cleaning"},
    {"query": "what PPE is required before entering the plant", "query_ar": "ما معدات الوقاية الشخصية المطلوبة قبل دخول المصنع؟",
     "golden_pages": [33, 34], "why": "PPE, Permit to Work"},
    {"query": "steps to shut down the unit in an emergency", "query_ar": "خطوات إيقاف الوحدة في حالات الطوارئ",
     "golden_pages": [28, 42], "why": "Emergency Shutdown System, Unit Shutdown Procedure"},
    {"query": "corrosion found on a pipeline during inspection", "query_ar": "تم العثور على تآكل في خط أنابيب أثناء الفحص",
     "golden_pages": [36, 37, 38], "why": "Corrosion Mechanisms, Inspection Techniques, Pipeline Integrity"},
    {"query": "compressor oil sample shows metal particles", "query_ar": "عينة زيت الضاغط تظهر جزيئات معدنية",
     "golden_pages": [5, 6], "why": "Compressors, Compressor Lubrication"},
]
possible_total = sum(len(q["golden_pages"]) for q in queries_with_answers)
print(f"{len(queries_with_answers)} queries, {possible_total} correct pages in total")

10 queries, 23 correct pages in total


One scoring function serves every method in the rest of the lab: a method is anything that turns a question into a ranked list of page numbers, best first.

In [16]:
def score_method(rank_fn, key="query"):
    """Correct pages found in the top 5, per query and in total. rank_fn(question) -> page numbers, best first."""
    per_query = []
    for q in queries_with_answers:
        top = list(rank_fn(q[key]))[:TOP_K]
        per_query.append(sum(1 for p in q["golden_pages"] if p in top))
    return per_query

def glove_ranker(page_vectors, tokenize_fn):
    def rank(question):
        q = page_vector(question, model, tokenize_fn).reshape(1, -1)
        scores = cosine_similarity(q, page_vectors)[0]
        return guide["page_number"].values[scores.argsort()[::-1]]
    return rank

results = pd.DataFrame({"query": [q["query"] for q in queries_with_answers],
                        "GloVe raw": score_method(glove_ranker(raw_vectors, simple_tokenize)),
                        "GloVe clean": score_method(glove_ranker(clean_vectors, clean_tokenize))})
results.loc["TOTAL"] = ["", results["GloVe raw"].sum(), results["GloVe clean"].sum()]
results

,query,GloVe raw,GloVe clean
0,pump seal is leaking oil,0,0
1,H2S gas detected in the area what to do,2,1
2,how to calibrate a pressure transmitter,1,1
3,relief valve opened and gas is going to flare,2,2
4,electric motor keeps tripping the breaker,1,1
5,heat exchanger pressure drop is increasing,1,1
6,what PPE is required before entering the plant,1,2
7,steps to shut down the unit in an emergency,2,2
8,corrosion found on a pipeline during inspection,2,2
9,compressor oil sample shows metal particles,1,1


## Step 8: A Keyword Baseline: TF-IDF

TF-IDF scores a page highly when it contains the question's words, weighting words that appear on few pages more than words found everywhere. It matches strings, not meanings, so it is the baseline a semantic method has to beat. `TfidfVectorizer` lowercases, splits and removes English stop words for us.

In [17]:
tfidf = TfidfVectorizer(stop_words="english")
page_tfidf = tfidf.fit_transform(guide["content"])

def tfidf_rank(question):
    scores = cosine_similarity(tfidf.transform([question]), page_tfidf)[0]
    return guide["page_number"].values[scores.argsort()[::-1]]

results.loc[results.index[:-1], "TF-IDF"] = score_method(tfidf_rank)
results.loc["TOTAL", "TF-IDF"] = results["TF-IDF"].iloc[:-1].sum()
results

,query,GloVe raw,GloVe clean,TF-IDF
0,pump seal is leaking oil,0,0,2.0
1,H2S gas detected in the area what to do,2,1,3.0
2,how to calibrate a pressure transmitter,1,1,1.0
3,relief valve opened and gas is going to flare,2,2,2.0
4,electric motor keeps tripping the breaker,1,1,1.0
5,heat exchanger pressure drop is increasing,1,1,2.0
6,what PPE is required before entering the plant,1,2,1.0
7,steps to shut down the unit in an emergency,2,2,2.0
8,corrosion found on a pipeline during inspection,2,2,3.0
9,compressor oil sample shows metal particles,1,1,2.0


## Step 9: Sentence Embeddings

Averaging word vectors throws away word order and treats every word alike. A **sentence-embedding model** is a pretrained network, trained on pairs of related sentences, that turns a whole text into one vector so that texts with similar meaning land close together. We encode each page once, encode the question, and rank by cosine similarity, exactly as before.

`all-MiniLM-L6-v2` is a small English model (about 23 million parameters, 384 numbers per text).

In [18]:
english_embedder = SentenceTransformer(ENGLISH_EMBEDDER)
page_emb_en = english_embedder.encode(guide["content"].tolist(), normalize_embeddings=True)
print(page_emb_en.shape)

def embed_ranker(embedder, page_embeddings, prefix=""):
    def rank(question):
        q = embedder.encode([prefix + question], normalize_embeddings=True)
        scores = (page_embeddings @ q.T)[:, 0]   # dot product of unit vectors = cosine similarity
        return guide["page_number"].values[scores.argsort()[::-1]]
    return rank

results.loc[results.index[:-1], "MiniLM (English)"] = score_method(embed_ranker(english_embedder, page_emb_en))
results.loc["TOTAL", "MiniLM (English)"] = results["MiniLM (English)"].iloc[:-1].sum()
results

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

(50, 384)


,query,GloVe raw,GloVe clean,TF-IDF,MiniLM (English)
0,pump seal is leaking oil,0,0,2.0,3.0
1,H2S gas detected in the area what to do,2,1,3.0,2.0
2,how to calibrate a pressure transmitter,1,1,1.0,2.0
3,relief valve opened and gas is going to flare,2,2,2.0,2.0
4,electric motor keeps tripping the breaker,1,1,1.0,2.0
5,heat exchanger pressure drop is increasing,1,1,2.0,2.0
6,what PPE is required before entering the plant,1,2,1.0,2.0
7,steps to shut down the unit in an emergency,2,2,2.0,2.0
8,corrosion found on a pipeline during inspection,2,2,3.0,3.0
9,compressor oil sample shows metal particles,1,1,2.0,2.0


**What to notice**

Compare the totals row. Preprocessing moves individual GloVe queries up and down, but it does not reliably raise the total: the clean tokenizer also throws away useful words (its `isalpha()` test deletes "h2s", as Step 11 shows). TF-IDF is strong whenever the question shares rare words with the right page (a page that names "H2S" or "transmitter"). The sentence-embedding model also finds pages that answer the question in different words. Look down each column for the queries a method misses: they are rarely the same queries for every method, which is why real search systems often combine a keyword score with an embedding score.

## Step 10: Ask in Arabic

Many technicians will type the question in Arabic, while the guide is in English. Here we run the 10 Arabic questions against the English pages with two embedding models:

- the English model from Step 9
- `multilingual-e5-small`, trained on text pairs in about 100 languages (118 million parameters, also 384 numbers per text). It expects the prefix `query: ` on questions and `passage: ` on documents.

TF-IDF and GloVe cannot take part: an Arabic question shares no words with an English page.

In [19]:
multi_embedder = SentenceTransformer(MULTILINGUAL_EMBEDDER)
page_emb_multi = multi_embedder.encode(["passage: " + t for t in guide["content"]], normalize_embeddings=True)

arabic = pd.DataFrame({"question (Arabic)": [q["query_ar"] for q in queries_with_answers],
                       "MiniLM (English)": score_method(embed_ranker(english_embedder, page_emb_en), key="query_ar"),
                       "multilingual-e5": score_method(embed_ranker(multi_embedder, page_emb_multi, "query: "), key="query_ar")})
arabic.loc["TOTAL"] = ["", arabic["MiniLM (English)"].sum(), arabic["multilingual-e5"].sum()]

# For comparison, the multilingual model on the English questions
english_with_multi = score_method(embed_ranker(multi_embedder, page_emb_multi, "query: "))
print(f"multilingual-e5, English questions: {sum(english_with_multi)}/{possible_total} correct pages in the top {TOP_K}")
print(f"multilingual-e5, Arabic questions:  {arabic.loc['TOTAL', 'multilingual-e5']}/{possible_total}")
arabic

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

model.safetensors: downloading bytes:           |  0.00B            

... (6 outputs left out)

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

multilingual-e5, English questions: 20/23 correct pages in the top 5
multilingual-e5, Arabic questions:  11/23


,question (Arabic),MiniLM (English),multilingual-e5
0,مانع التسرب في المضخة يسرب الزيت,0,1
1,تم رصد غاز كبريتيد الهيدروجين في المنطقة، ماذا...,0,2
2,كيف أعاير جهاز إرسال الضغط؟,0,1
3,انفتح صمام التنفيس والغاز يذهب إلى الشعلة,0,1
4,المحرك الكهربائي يفصل القاطع باستمرار,0,1
5,انخفاض الضغط في المبادل الحراري يزداد,0,0
6,ما معدات الوقاية الشخصية المطلوبة قبل دخول الم...,0,1
7,خطوات إيقاف الوحدة في حالات الطوارئ,0,2
8,تم العثور على تآكل في خط أنابيب أثناء الفحص,0,1
9,عينة زيت الضاغط تظهر جزيئات معدنية,0,1


**What to notice**

The English-only model has no idea what an Arabic question means: it finds almost none of the correct pages, fewer than a random ranking would find by chance. The multilingual model maps an Arabic question and an English page with the same meaning to nearby vectors, so it can answer Arabic questions from English documents: **cross-lingual retrieval**. Compare its Arabic and English totals to see how much is lost in crossing languages.

## Step 11: Look at One Query in Detail

Change `QUERY_NUMBER` below to explore different queries (1 to 10). Look at which pages each method gets right, and at the words the GloVe methods actually used.

In [20]:
QUERY_NUMBER = 2       # change this (1 to 10)

q = queries_with_answers[QUERY_NUMBER - 1]
print(f'Query: "{q["query"]}"  |  Arabic: "{q["query_ar"]}"')
print(f'Correct pages: {q["golden_pages"]} ({q["why"]})\n')

methods = {"GloVe raw": (glove_ranker(raw_vectors, simple_tokenize), "query"),
           "GloVe clean": (glove_ranker(clean_vectors, clean_tokenize), "query"),
           "TF-IDF": (tfidf_rank, "query"),
           "MiniLM (English)": (embed_ranker(english_embedder, page_emb_en), "query"),
           "multilingual-e5, Arabic question": (embed_ranker(multi_embedder, page_emb_multi, "query: "), "query_ar")}
for name, (rank_fn, key) in methods.items():
    top = list(rank_fn(q[key]))[:TOP_K]
    marks = [f"{p}{'*' if p in q['golden_pages'] else ''}" for p in top]
    print(f"{name:34s} top {TOP_K}: {', '.join(marks)}   (* = correct)")

print("\nWords used for matching by GloVe")
print("   Raw:    ", simple_tokenize(q["query"]))
print("   Cleaned:", clean_tokenize(q["query"]))

Query: "H2S gas detected in the area what to do"  |  Arabic: "تم رصد غاز كبريتيد الهيدروجين في المنطقة، ماذا أفعل؟"
Correct pages: [30, 29, 33] (H2S Safety, Fire and Gas Detection, PPE)

GloVe raw                          top 5: 29*, 30*, 32, 28, 44   (* = correct)
GloVe clean                        top 5: 29*, 6, 32, 22, 16   (* = correct)
TF-IDF                             top 5: 30*, 29*, 16, 18, 33*   (* = correct)
MiniLM (English)                   top 5: 30*, 29*, 16, 17, 18   (* = correct)
multilingual-e5, Arabic question   top 5: 17, 30*, 29*, 18, 16   (* = correct)

Words used for matching by GloVe
   Raw:     ['h2s', 'gas', 'detected', 'in', 'the', 'area', 'what', 'to', 'do']
   Cleaned: ['gas', 'detected', 'area']


## Summary

- Search by meaning means turning pages and questions into vectors and ranking pages by cosine similarity.
- Averaged word vectors are a weak page representation, and preprocessing is a trade-off: it removes noise words but can delete the one word that mattered.
- TF-IDF is a strong keyword baseline whenever questions and pages share rare words.
- Sentence-embedding models, trained for similarity, also match pages that answer the question in other words.
- A multilingual embedding model can retrieve English pages for Arabic questions; an English-only one cannot.

## Exercises

1. Which method has the **highest total** on the English questions? Is it the best on every query?
2. Without looking at the clean tokenizer above, write your own version that also removes words shorter than three letters. Does the GloVe clean total change?
3. Set `QUERY_NUMBER = 2` (H2S query). How many words did preprocessing remove? Were those words useful or just noise?
4. We **average** word vectors into one vector. Does *"pump failed"* get the same vector as *"failed pump"*? Does a sentence-embedding model treat them the same?
5. If a technician searches *"the unit is not running"*, the word *"not"* is a stop word and gets removed. Could that be a problem for TF-IDF and the clean GloVe method?
6. Write two questions of your own in Gulf or Egyptian dialect for pages of your choice. Does the multilingual model still find them?

## Further Reading

- [Pennington et al., 2014. GloVe: Global Vectors for Word Representation](https://aclanthology.org/D14-1162/)
- [Reimers and Gurevych, 2019. Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks](https://arxiv.org/abs/1908.10084)
- [Wang et al., 2024. Multilingual E5 Text Embeddings: A Technical Report](https://arxiv.org/abs/2402.05672)
- [Thakur et al., 2021. BEIR: A Heterogenous Benchmark for Zero-shot Evaluation of Information Retrieval Models](https://arxiv.org/abs/2104.08663)

### Contributed by: Yazan alshoibi